# 14 — Publication figures from stored results

**Purpose.** Figures for the manuscript built from the stored full-scale experiment results in experiments/results/ (no re-running).

**Author:** Guy M. Kaptue T. · **Execution:** independent, seeded (`SEED = 0`), demonstration scale (CPU, about a minute). Outputs: `reports/figures/14_publication_figures_*.png|pdf` and `reports/notebooks/14_publication_figures.json`.

> **Data statement.** SYN-IDS-v1 is completely synthetic despite its name (scikit-learn `make_classification`); it is not real intrusion data and not patient data. The public healthcare datasets are de-identified research tables; no real hospital data is used, all federated clients are simulated locally in one process, and raw source files are not redistributed.

In [1]:
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
def _root():
    p = Path(os.environ.get("FL_ROOT") or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "pyproject.toml").exists() and (q / "src" / "federated_learning").exists():
            return q
    raise RuntimeError("repository root not found; set FL_ROOT")
ROOT = _root(); sys.path.insert(0, str(ROOT / "src")); os.environ["FL_ROOT"] = str(ROOT)
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
SEED = 0; np.random.seed(SEED)
NB_ID = "14_publication_figures"
FIG = ROOT / "reports" / "figures"; OUT = ROOT / "reports" / "notebooks"; FIG.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
PAL = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]   # Okabe-Ito, colour-blind safe
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": matplotlib.cycler(color=PAL), "axes.grid": True, "grid.alpha": .25, "legend.frameon": False})
SUMMARY = {}
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{NB_ID}_{name}.{ext}", bbox_inches="tight")
    plt.show()
def save_summary():
    (OUT / f"{NB_ID}.json").write_text(json.dumps(SUMMARY, indent=2, default=float)); print("summary ->", OUT / f"{NB_ID}.json")

## 1. Load stored results

In [2]:
R_ = ROOT / "experiments" / "results"
perf = pd.read_csv(R_ / "fl_perf_002" / "task_2_5_kpi_check.csv"); byz = pd.read_csv(R_ / "fl_attack_003" / "task_3_2_summary.csv")
display(perf.head()); display(byz.head())

,config,n_clients,training_overhead_pct,target_lt_30,comm_overhead_pct,target_lt_40,accuracy_loss_pp,target_lt_1
0,he_ckks,10,134.680022,False,2956.152304,False,0.0,True
1,he_ckks,50,415.678113,False,2955.823839,False,0.0,True
2,he_ckks,100,751.329194,False,2956.461901,False,0.0,True
3,he_ckks,500,1747.659374,False,2955.432692,False,0.0,True
4,secagg_complete,10,29.965899,True,89.477504,False,0.0,True


,mal_frac,behavior,aggregator,acc,acc_ci,loss,agg_ms
0,0.1,random,mean,0.773611,"[0.770, 0.777]",0.477327,0.251915
1,0.1,random,median,0.777963,"[0.772, 0.784]",0.470114,0.982770
2,0.1,random,trimmed_mean,0.777778,"[0.771, 0.784]",0.470354,0.463034
3,0.1,random,krum,0.774167,"[0.766, 0.782]",0.474071,0.494733
4,0.1,random,multi_krum,0.780093,"[0.776, 0.784]",0.466505,0.503698


## 2. Figure: secure-aggregation overhead and robust aggregation under attack

In [3]:
fig, ax = plt.subplots(1, 2, figsize=(7.4, 3))
for cfg_, g in perf.groupby("config"): ax[0].plot(g.n_clients, g.training_overhead_pct, "o-", label=cfg_)
ax[0].axhline(30, ls="--", color="grey"); ax[0].set_xscale("log"); ax[0].set_xlabel("clients"); ax[0].set_ylabel("training overhead (%)"); ax[0].set_title("Overhead vs 30% budget"); ax[0].legend(fontsize=7)
b = byz[(byz.behavior == "sign_flip") & (byz.mal_frac == byz.mal_frac.max())]
ax[1].bar(b.aggregator, b.acc, color=PAL[0]); ax[1].set_ylabel("accuracy"); ax[1].set_title(f"Sign-flip, {byz.mal_frac.max():.0%} malicious"); plt.setp(ax[1].get_xticklabels(), rotation=20)
save_fig(fig, "overhead_and_robustness"); SUMMARY["source_files"] = ["fl_perf_002/task_2_5_kpi_check.csv", "fl_attack_003/task_3_2_summary.csv"]; save_summary()

summary -> /Users/guykaptue/Downloads/federated_learning_security/reports/notebooks/14_publication_figures.json


## Limitations
Demonstration scale (few rounds, one seed unless stated): numbers here illustrate the workflow and are not the study's reported results, which come from the full experiment pipeline in `experiments/results/` with multiple seeds and confidence intervals.